In [1]:
!python --version

Python 3.13.7


# SynTagRus

In [ ]:
import os
import pandas as pd
from typing import Dict, Tuple, Any
from datetime import datetime
from enum import Enum
import re
import json
import xml.etree.ElementTree as ET
from alite_backend.db import models, schemas

In [3]:
corpus_location = "./raw/SynTagRus2022/"

In [4]:
bodyTextDf_loc = './data/bodyTextDf.json'
bodyLibDf_loc = './data/bodyLibDf.json'
infDict_loc = './data/infDict.json'
infDictDf_loc = './data/infDictDf.json'

# Make

In [5]:
feat_dict = {
    "pos" : ["S", "A", "V", "ADV", "NUM", "PR", "COM", "CONJ", "P", "PART", "INTJ", "NID"],
    "noun_animacy" : ["ОД", "НЕОД"],
    "gram_gender" : ["МУЖ", "ЖЕН", "СРЕД"],
    "gram_number" : ["ЕД", "МН"],
    "subst_case" : ["ИМ", "РОД", "ПАРТ", "ДАТ", "ВИН", "ТВОР", "ПР", "МЕСТН"],
    "alt_adjv_type" : ["СРАВ", "ПРЕВ", "КР"],
    "verb_rep" : ["ИНФ", "ПРИЧ", "ДЕЕПР"],
    "verb_mood" : ["ИЗЪЯВ", "ПОВ"],
    "verb_aspect" : ["НЕСОВ", "СОВ"],
    "conj_person" : ["1-Л", "2-Л", "3-Л"],
    "passive" : ["СТРАД"],
    "word_formation" : ["СЛ"],
    "mod_comparative" : ["СМЯГ"]
    }

In [6]:
feat_def_dict = {
    "S": ("pos", models.EnumPartOfSpeech.NOUN),
    "A": ("pos", models.EnumPartOfSpeech.ADJECTIVE),
    "V": ("pos", models.EnumPartOfSpeech.VERB),
    "ADV": ("pos", models.EnumPartOfSpeech.ADVERB),
    "NUM": ("pos", models.EnumPartOfSpeech.NUMERAL),
    "PR": ("pos", models.EnumPartOfSpeech.PREPOSITION),
    "COM": ("pos", models.EnumPartOfSpeech.COM),
    "CONJ": ("pos", models.EnumPartOfSpeech.CONJUNCTION),
    "P": ("pos", models.EnumPartOfSpeech.PRONOUN),
    "PART": ("pos", models.EnumPartOfSpeech.PARTICLE),
    "INTJ": ("pos", models.EnumPartOfSpeech.INTERJECTION),
    "NID": ("pos", models.EnumPartOfSpeech.UNKNOWN),
    "ОД": ("noun_animacy", True),
    "НЕОД": ("noun_animacy", False),
    "МУЖ": ("gram_gender", models.EnumGramGender.MASCULINE),
    "ЖЕН": ("gram_gender", models.EnumGramGender.FEMININE),
    "СРЕД": ("gram_gender", models.EnumGramGender.NEUTER),
    "ЕД": ("gram_number", models.EnumGramNum.SINGULAR),
    "МН": ("gram_number", models.EnumGramNum.PLURAL),
    "ИМ": ("subst_case", models.EnumSubstCase.NOMINATIVE),
    "РОД": ("subst_case", models.EnumSubstCase.GENITIVE),
    "ПАРТ": ("subst_case", models.EnumSubstCase.PARTITIVE),
    "ДАТ": ("subst_case", models.EnumSubstCase.DATIVE),
    "ВИН": ("subst_case", models.EnumSubstCase.ACCUSATIVE),
    "ТВОР": ("subst_case", models.EnumSubstCase.INSTRUMENTAL),
    "ПР": ("subst_case", models.EnumSubstCase.PREPOSITIONAL),
    "МЕСТН": ("subst_case", models.EnumSubstCase.LOCATIVE),
    "СРАВ": ("alt_adjv_type", models.EnumAltAdjvType.COMPARATIVE),
    "ПРЕВ": ("alt_adjv_type", models.EnumAltAdjvType.SUPERLATIVE),
    "КР": ("alt_adjv_type", models.EnumAltAdjvType.SHORT),
    "ИНФ": ("verb_infinitive", True),
    "ИЗЪЯВ": ("verb_mood", models.EnumVerbMood.INDICATIVE),
    "ПОВ": ("verb_mood", models.EnumVerbMood.IMPERATIVE),
    "НЕСОВ": ("verb_aspect", models.EnumVerbAspect.IMPERFECTVE),
    "СОВ": ("verb_aspect", models.EnumVerbAspect.PERFECTIVE),
    "1-Л": ("verb_conj_person", models.EnumConjPerson.FIRST),
    "2-Л": ("verb_conj_person", models.EnumConjPerson.SECOND),
    "3-Л": ("verb_conj_person", models.EnumConjPerson.THIRD),
    # "СТРАД": ("verb_trans_refl", models.EnumVerbTransRefl.REFLEXIVE),
    "ПРИЧ": ("part_type", models.EnumPartType.ADJECTIVAL),
    "ДЕЕПР": ("part_type", models.EnumPartType.ADVERBIAL),
    # "other": ["СЛ", "СМЯГ"],
}

In [7]:
def parse_features(feat_string: str) -> Dict[str, Any]:
    """
    Parses a space-separated feature string into a dictionary based on feat_def_dict.
    O(N) complexity where N is the number of traits per word.
    """
    if not feat_string:
        return {}
        
    parsed_traits = {}
    for code in feat_string.split():
        mapping = feat_def_dict.get(code)
        if mapping:
            col_name, val = mapping
            # Use .value if using Enums so it's DB-ready (e.g., "NOUN" instead of <POS.NOUN>)
            parsed_traits[col_name] = val.value if isinstance(val, Enum) else val
            
    return parsed_traits

In [ ]:
import xml.etree.ElementTree as ET
from typing import Dict, Any, Tuple, List
# from mappings import parse_features  # Assuming your feat_def_dict logic lives here

def parse_tgt_file(file_path: str) -> Tuple[Dict[str, Any], List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Parses a SynTagRus .tgt file into native Python dictionaries.
    Bypasses Pandas entirely for optimal ETL performance into SQL databases.
    
    Returns:
        doc_data: Dict containing document-level metadata.
        sentences_data: List of Dicts containing sentence data.
        tokens_data: List of Dicts containing word/token data.
    """
    # initialize the XML parser
    tree = ET.parse(file_path)
    root = tree.getroot()

    # extract document metadata (<inf> tag)
    inf_node = root.find('./inf')
    
    # safely extract text using findtext. defaults to None if the tag is missing.
    title = inf_node.findtext('title')
    author = inf_node.findtext('author')
    source = inf_node.findtext('source')
    
    date_str = inf_node.findtext('date')
    
    doc_data = {
        "title": title,
        "author": author,
        "source": source,
        "date": date_str
    }

    # --- Extract Sentences and Tokens ---
    sentences_data = []
    tokens_data = []

    # Iterate over all <S> (sentence) elements
    for s_node in root.findall('.//S'):
        # SynTagRus sequential ID
        sentence_index = int(s_node.get('ID', 0))
        
        # .itertext() grabs all raw text recursively, bypassing the <W> XML nodes
        # This gives us the clean, readable sentence.
        raw_text = "".join(s_node.itertext()).strip()
        
        sentences_data.append({
            # document_id will be injected in the load script after Doc insertion
            "sentence_index": sentence_index,
            "raw_text": raw_text
        })

        # Iterate over all <W> (word) elements within the current sentence
        for w_node in s_node.findall('W'):
            
            # SynTagRus dependency trees map the root word as '_root'. 
            # A PostgreSQL Integer column requires an actual integer or NULL.
            raw_dom = w_node.get('DOM')
            head_index = int(raw_dom) if raw_dom and raw_dom != '_root' else None
            
            # Apply your transformation mapping to the FEAT string
            features = parse_features(w_node.get('FEAT', ''))
            
            tokens_data.append({
                # We store sentence_index temporarily so the load script knows 
                # which sentence this word belongs to.
                "sentence_index": sentence_index, 
                "token_index": int(w_node.get('ID', 0)),
                "lexeme_raw": w_node.text.strip() if w_node.text else "",
                "lemma_raw": w_node.get('LEMMA'),
                "head_index": head_index,
                "dep_rel": w_node.get('LINK'),
                "semantic_tag": w_node.get('KSNAME'),
                "features": features 
            })

    return doc_data, sentences_data, tokens_data

In [9]:
enum_feat_dict = {}
for key, val in feat_dict.items():
    enum_feat = {key: dict(enumerate(val))}
    enum_feat_dict.update(enum_feat)
#enum_feat_dict
rev_enum_feat_dict = {}
for key, val in enum_feat_dict.items():
    val_dict = {}
    for k, v in val.items():
        val_dict.update({v:k})
    rev_enum_feat_dict.update({key:val_dict})
#rev_enum_feat_dict

In [10]:
def list_files_recursive(path='.', ff=None, file_ext=".tgt", re_pattern=False, sort=False):

    # instantiate list for all files
    file_list = []
    
    def filter_files(path):
        for root, dirs, files in os.walk(path):
            # Remove unwanted dirs in-place
            dirs[:] = [d for d in dirs if not d.startswith('.') and d != '.ipynb_checkpoints']
        
            for f in files:
                if f.endswith(file_ext) and not f.startswith("."):
                    full_path = os.path.join(root, f)
                    file_list.append(full_path)
        
    if ff:
        filter_files(path)
    else:
        # for loop through the dirs
        for root, dirs, files in os.walk(path):
            for file in files:
                # check for existence of RE pattern to be applied
                if re_pattern != None:
                    f = re.compile(re_pattern)
                    if f.search(file):
                        file_list.append(os.path.join(root, file))
                else:
                    file_list.append(os.path.join(root, file))

    if sort:
        file_list.sort()
    
    return file_list

In [ ]:
def parseW(file_list):
    
    #
    parsedBodyDf = pd.DataFrame(columns=['doc_id'])

    #
    parsedInfDict = {}
    
    #
    idx = 0
    
    #
    print(len(file_list))
    
    # function to assign codes to category columns
    def assign_codes(code_list):
        result = {cat: [] for cat in feat_dict.keys()}
        for code in code_list:
            category = code_to_category.get(code)
            if category:
                result[category].append(code)
        # Join multiple codes (if any) into a space-separated string
        return {cat: ' '.join(codes) if codes else None for cat, codes in result.items()}
    
    for file in file_list:
        # make dictionary from 'inf' tag contents (author, date, source, title)
        parsedInf = pd.read_xml(file, xpath="/text/inf").T[0].to_dict()
        
        #
        wDf = pd.read_xml(file, xpath="/text/body/S/W")
        
        #
        Slines = wDf.loc[wDf.ID == 1].index
        
        #
        wDf.columns = [x.lower() for x in wDf.columns]
        
        #
        wDf = wDf.rename(columns={'id':'word_id'})
        
        #
        wDf.loc[wDf.loc[wDf.word_id == 1].index, 'sent_id'] = range(1,len(Slines)+1)
        
        #
        wDf.sent_id = wDf.sent_id.ffill().astype('int')
        
        #
        wDf.lemma = wDf.lemma.apply(lambda x: x.lower())
    
        # invert the dictionary to map code to category
        code_to_category = {}
        for cat, codes in feat_dict.items():
            for code in codes:
                code_to_category[code] = cat
        
        # apply the function to each row, collect the new columns
        new_cols = wDf['feat'].str.split().apply(assign_codes).apply(pd.Series)
        
        # concatenate the original df with new_cols
        wDf = pd.concat([wDf, new_cols], axis=1)
        
        # drop original 'feat' column
        wDf = wDf.drop(columns=['feat'])
        
        # add idx to df
        wDf.loc[:, 'doc_id'] = idx
        
        # give the index a name: t(oken)_id
        wDf.index.name = 'doc_tok_id'
        
        # reset index for workability
        wDf = wDf.reset_index()

        #
        parsedInfDict[idx] = parsedInf

        #
        parsedBodyDf = pd.concat([parsedBodyDf, wDf], axis=0, ignore_index=True)
        
        #
        idx += 1

    
    parsedBodyLibDf = parsedBodyDf[[
        'doc_id', 'sent_id', 'word_id', 'doc_tok_id', 'dom', 'link', 'lemma',
        'ksname', 'nodetype', 'extracomm', 'status'
    ]]

    parsedBodyTextDf = parsedBodyDf[[
        'doc_id', 'doc_tok_id', 'w', 'pos', 'animacy', 'gender', 'number', 
        'case', 'adjective_level', 'shortness', 'verb_rep', 'mood', 
        'aspect', 'person', 'passive', 'word_formation', 'mod_comparative'
    ]]    
    
    return parsedInfDict, parsedBodyLibDf, parsedBodyTextDf

In [ ]:
def parseW_optimized(file_list, feat_dict):
    """
    Parses a list of XML files into DataFrames in a more optimized way.

    Args:
        file_list (list): A list of file paths to the XML files.
        feat_dict (dict): A dictionary mapping feature categories to their codes.

    Returns:
        tuple: A tuple containing (parsed_inf_dict, parsedBodyLibDf, parsedBodyTextDf).
    """
    # --- 1. Pre-computation (Moved outside the loop) ---
    # Invert the dictionary once before the loop begins.
    code_to_category = {
        code: category for category, codes in feat_dict.items() for code in codes
    }

    # Helper function is also defined once.
    def assign_codes(code_list):
        if not isinstance(code_list, list):
            return {cat: None for cat in feat_dict.keys()}
        
        result = {cat: [] for cat in feat_dict.keys()}
        for code in code_list:
            category = code_to_category.get(code)
            if category:
                result[category].append(code)
        return {cat: ' '.join(codes) if codes else None for cat, codes in result.items()}

    # --- 2. Process files and collect DataFrames in a list ---
    all_body_dfs = []
    parsed_inf_dict = {}

    print(f"Processing {len(file_list)} files...")
    for doc_id, file in enumerate(file_list):
        # Process metadata
        parsed_inf = pd.read_xml(file, xpath="/text/inf").T[0].to_dict()
        parsed_inf_dict[doc_id] = parsed_inf

        # Process main content from files like A_on_myatezhnyi.tgt
        wDf = pd.read_xml(file, xpath="/text/body/S/W")
        
        # --- 3. Vectorized and Chained Operations ---
        wDf.columns = wDf.columns.str.lower()
        wDf = wDf.rename(columns={'id': 'word_id'})
        
        # Calculate sentence IDs more efficiently using cumsum()
        sent_starts = wDf['word_id'] == 1
        wDf['sent_id'] = sent_starts.cumsum()
        
        wDf['lemma'] = wDf['lemma'].str.lower()

        # Process the 'feat' column
        if 'feat' in wDf.columns:
            new_cols_df = wDf['feat'].str.split().apply(assign_codes).apply(pd.Series)
            wDf = pd.concat([wDf, new_cols_df], axis=1)
            wDf = wDf.drop(columns=['feat'])

        wDf['doc_id'] = doc_id
        all_body_dfs.append(wDf)

    # --- 4. Single Concatenation After the Loop ---
    if not all_body_dfs:
        return {}, pd.DataFrame(), pd.DataFrame()
        
    parsedBodyDf = pd.concat(all_body_dfs, ignore_index=True)
    parsedBodyDf = parsedBodyDf.reset_index().rename(columns={'index': 'doc_tok_id'})

    # --- 5. Final DataFrame Slicing ---
    lib_cols = [
        'doc_id', 'sent_id', 'word_id', 'doc_tok_id', 'dom', 'link', 'lemma',
        'ksname', 'nodetype', 'extracomm', 'status'
    ]
    text_cols = [
        'doc_id', 'doc_tok_id', 'w', 'pos', 'animacy', 'gender', 'number', 
        'case', 'adjective_level', 'shortness', 'verb_rep', 'mood', 
        'aspect', 'person', 'passive', 'word_formation', 'mod_comparative'
    ]
    
    # Filter columns to only those that exist to avoid KeyErrors
    parsedBodyLibDf = parsedBodyDf[[col for col in lib_cols if col in parsedBodyDf.columns]]
    parsedBodyTextDf = parsedBodyDf[[col for col in text_cols if col in parsedBodyDf.columns]]
    
    return parsed_inf_dict, parsedBodyLibDf, parsedBodyTextDf

In [11]:
file_list = list_files_recursive(corpus_location, ff=True, sort=True)

In [12]:
sentences_df, words_df = parse_tgt_file(file_list[0], 1)

In [13]:
sentences_df

,doc_id,sent_id,raw_text
0,1,1,"""А \nОН, \nМЯТЕЖНЫЙ, \nПРОСИТ \nБУРИ…"""
1,1,2,Что \nтакое \nсудьба?
2,1,3,"Может, \nэто \nто, \nчто \nнаписано \nне \nна ..."
3,1,4,Клиффорд \nСаймак.
4,1,5,"""Кукла \nсудьбы"""
...,...,...,...
143,1,144,Она \nскорее - \nмножество \nпотенций.
144,1,145,Какая \nчасть \nиз \nмножества \nпотенций \nбу...
145,1,146,Но \nведь \nи \nбиография \nв значительной мер...
146,1,147,"Из \n""репертуара \nвозможных \nответов \nна \n..."


In [14]:
words_df

,doc_id,sent_id,word_id,dom,lemma,w,pos,gram_number,gram_gender,subst_case,noun_animacy,verb_aspect,verb_mood,verb_conj_person,part_type,alt_adjv_type,verb_infinitive
0,1,1,1,_root,А,А,conjunction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,2,4,ОН,ОН,noun,singular,masculine,nominative,True,NaN,NaN,NaN,NaN,NaN,NaN
2,1,1,3,2,МЯТЕЖНЫЙ,МЯТЕЖНЫЙ,adjective,singular,masculine,nominative,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,1,4,1,ПРОСИТЬ,ПРОСИТ,verb,singular,NaN,NaN,NaN,imperfective,indicative,third-person,NaN,NaN,NaN
4,1,1,5,4,БУРЯ,БУРИ,noun,singular,feminine,genitive,False,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2611,1,148,16,14,КАК БУДТО,Как будто,conjunction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2612,1,148,17,19,В,в,preposition,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2613,1,148,18,17,БУРЯ,бурях,noun,plural,feminine,prepositional,False,NaN,NaN,NaN,NaN,NaN,NaN
2614,1,148,19,16,БЫТЬ,есть,verb,singular,NaN,NaN,NaN,imperfective,indicative,third-person,NaN,NaN,NaN


In [ ]:
parsedInfDict, parsedBodyLibDf, parsedBodyTextDf = parseW_optimized(file_list, feat_dict)

In [ ]:
parsedBodyTextDf

parsedBodyTextDf.to_json(path_or_buf=bodyTextDf_loc, orient='index')

In [ ]:
parsedBodyLibDf

parsedBodyLibDf.to_json(path_or_buf=bodyLibDf_loc, orient='index')

In [ ]:
parsedInfDict

pd.DataFrame(parsedInfDict).T.to_json(path_or_buf=infDictDf_loc, orient='index')

# Explore

In [ ]:
parsedBodyLibDf = pd.read_json(bodyLibDf_loc)

In [ ]:
parsedBodyLibDf

In [ ]:
parsedBodyTextDf = pd.read_json(bodyTextDf_loc).T

In [ ]:
parsedBodyTextDf.loc[parsedBodyTextDf['pos'] == 'COM']

In [ ]:
parsed_info_df = pd.read_json(infDictDf_loc)

In [ ]:
parsed_info_df.T

### sentence_docs

In [ ]:
sentenceDocsDf = pd.DataFrame(parsedInfDict).T
sentenceDocsDf

### sentence_token_meta

In [ ]:
parsedBodyLibDf['doc_tok_id'] = parsedBodyLibDf['doc_tok_id'].astype('Int64')
parsedBodyLibDf['word_id'] = parsedBodyLibDf['word_id'].astype('Int64')
parsedBodyLibDf['sent_id'] = parsedBodyLibDf['sent_id'].astype('Int64')
parsedBodyLibDf

In [ ]:
parsedBodyLibDf[['doc_id', 'sent_id']].drop_duplicates().reset_index().drop('index', axis=1)

### sentence_token_gram

In [ ]:
parsedBodyTextDf['doc_tok_id'] = parsedBodyTextDf['doc_tok_id'].astype('Int64')
parsedBodyTextDf

### sentence_tokens

In [ ]:
#sentenceTokensDf = pd.DataFrame(parsedBodyTextDf.w.unique())
sentenceTokensDf = pd.DataFrame(parsedBodyTextDf.w.str.lower().unique())
sentenceTokensDf

In [ ]:
with open(infDict_loc, "w") as file:
    json.dump(parsedInfDict, file, indent=4)

In [ ]:
# Open and read the JSON file
with open(infDict_loc, 'r') as file:
    parsedInfDict = json.load(file)

In [ ]:
parsedBodyLibDf = pd.read_json(bodyLibDf_loc)

In [ ]:
parsedBodyTextDf = pd.read_json(bodyTextDf_loc)

In [ ]:
parsedBodyLibDf.T

In [ ]:
parsedBodyTextDf = parsedBodyTextDf.T

In [ ]:
parsedBodyTextDf.loc[parsedBodyTextDf.lemma == "человек"].sample(5)